# 01 — What We Built

Before moving on to the library version, we take stock.

Good engineers do not just ship and move on — they understand what they built, where it breaks, and what it would cost to fix those things. This notebook measures the handbuilt system honestly.

## What We Built — Component Inventory

| Component | Where | What it does |
|---|---|---|
| Douglas-Peucker | Module 01 | Reduces point count per line segment |
| LOD pipeline | Module 02 | Produces 4 simplified GeoJSON files |
| Bbox computation | Module 03 | Gets the extent of any feature |
| Bbox intersection | Module 03 | Tests if a feature overlaps the viewport |
| Uniform grid index | Module 04 | Buckets features for fast viewport queries |
| LOD decision function | Module 05 | Selects the right file by zoom level |
| Live map viewer | Module 06 | Wires everything into an interactive display |

Each component was built from scratch. We understand every line.

## Measuring the System

In [1]:
import json
import math
import time
from pathlib import Path

# Single-file version: use only data/ne_10m_railroads.geojson.
# No data/lod folder and no railroads_*.geojson files are required.

def find_raw_path():
    cwd = Path.cwd().resolve()
    for base in [cwd] + list(cwd.parents):
        candidate = base / "data" / "ne_10m_railroads.geojson"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/ne_10m_railroads.geojson. "
        "Make sure the notebook is inside the project folder that contains the data folder."
    )

raw_path = find_raw_path()

LOD_CONFIG = {
    "coarse":     {"epsilon": 1.00, "max_scalerank": 4},
    "medium":     {"epsilon": 0.50, "max_scalerank": None},
    "fine":       {"epsilon": 0.15, "max_scalerank": None},
    "extra_fine": {"epsilon": 0.03, "max_scalerank": None},
}


def iter_points(coords):
    """Yield [lon, lat] points from LineString or MultiLineString coordinates."""
    if not coords:
        return
    if isinstance(coords[0], (int, float)):
        yield coords
    else:
        for part in coords:
            yield from iter_points(part)


def geometry_point_count(geometry):
    return sum(1 for _ in iter_points(geometry["coordinates"]))


def point_distance_to_segment(p, a, b):
    px, py = p
    ax, ay = a
    bx, by = b
    dx = bx - ax
    dy = by - ay

    if dx == 0 and dy == 0:
        return math.hypot(px - ax, py - ay)

    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)
    t = max(0, min(1, t))
    proj_x = ax + t * dx
    proj_y = ay + t * dy
    return math.hypot(px - proj_x, py - proj_y)


def douglas_peucker(points, epsilon):
    """Pure-Python Douglas-Peucker simplification for one line."""
    if len(points) <= 2:
        return points

    start = points[0]
    end = points[-1]
    max_dist = -1
    split_index = 0

    for i in range(1, len(points) - 1):
        dist = point_distance_to_segment(points[i], start, end)
        if dist > max_dist:
            max_dist = dist
            split_index = i

    if max_dist > epsilon:
        left = douglas_peucker(points[:split_index + 1], epsilon)
        right = douglas_peucker(points[split_index:], epsilon)
        return left[:-1] + right

    return [start, end]


def simplify_geometry(geometry, epsilon):
    geom_type = geometry["type"]
    coords = geometry["coordinates"]

    if geom_type == "LineString":
        return {"type": "LineString", "coordinates": douglas_peucker(coords, epsilon)}

    if geom_type == "MultiLineString":
        return {
            "type": "MultiLineString",
            "coordinates": [douglas_peucker(part, epsilon) for part in coords if len(part) >= 2],
        }

    return geometry


def get_scalerank(feature):
    try:
        return int(feature.get("properties", {}).get("scalerank", 99))
    except Exception:
        return 99


def build_lod_features(raw_features, epsilon, max_scalerank=None):
    """Build one LOD in memory from the original feature list."""
    lod_features = []

    for feature in raw_features:
        if max_scalerank is not None and get_scalerank(feature) > max_scalerank:
            continue

        lod_features.append({
            "type": "Feature",
            "properties": dict(feature.get("properties", {})),
            "geometry": simplify_geometry(feature["geometry"], epsilon),
        })

    return lod_features


def load_raw_features():
    with open(raw_path) as f:
        return json.load(f)["features"]


def build_all_lods_from_raw(raw_features):
    return {
        name: build_lod_features(raw_features, **cfg)
        for name, cfg in LOD_CONFIG.items()
    }

print("Using raw file:", raw_path)
print(f"{'LOD':<28} {'Est. size (MB)':>14} {'Features':>10} {'Total pts':>12} {'Build (s)':>10}")
print("-" * 82)

t0 = time.perf_counter()
raw_features = load_raw_features()
raw_load_time = time.perf_counter() - t0
raw_size = raw_path.stat().st_size / 1_000_000
raw_points = sum(geometry_point_count(f["geometry"]) for f in raw_features)
print(f"{'original':<28} {raw_size:>14.2f} {len(raw_features):>10,} {raw_points:>12,} {raw_load_time:>10.3f}")

lod_features_by_name = {}
for name, cfg in LOD_CONFIG.items():
    t0 = time.perf_counter()
    features = build_lod_features(raw_features, **cfg)
    build_time = time.perf_counter() - t0
    lod_features_by_name[name] = features

    fc = {"type": "FeatureCollection", "features": features}
    estimated_size = len(json.dumps(fc).encode("utf-8")) / 1_000_000
    total_points = sum(geometry_point_count(f["geometry"]) for f in features)

    print(f"{name:<28} {estimated_size:>14.2f} {len(features):>10,} {total_points:>12,} {build_time:>10.3f}")



Using raw file: /workspaces/ricardoayala2510-Spatial-Data-Mapping/assigments completed/03-Data_Manager/data/ne_10m_railroads.geojson
LOD                          Est. size (MB)   Features    Total pts  Build (s)
----------------------------------------------------------------------------------
original                              39.60     25,413    1,396,480      2.060
coarse                                 1.08      2,845        5,690      0.555
medium                                 9.55     25,413       50,969      2.762
fine                                   9.61     25,413       53,182      2.292
extra_fine                            10.15     25,413       75,577      3.644


## Where the System Still Hurts

The viewer works. But it has real limitations. Let's name them honestly.

### Pain Point 1 — Startup Cost

Every session, we load 4 files and build 4 grid indexes. This takes several seconds before the map is usable.

A tile server has no startup cost — tiles are pre-built and stored. The server just reads a file from a database and sends it.

In [2]:
def feature_bbox(feature):
    points = list(iter_points(feature["geometry"]["coordinates"]))
    lons = [p[0] for p in points]
    lats = [p[1] for p in points]
    return [min(lons), min(lats), max(lons), max(lats)]

class GridIndex:
    CELL_SIZE = 10.0
    def __init__(self):
        self.cells = {}
    def _cells(self, bbox):
        lo, la, hi, ha = bbox
        cs = self.CELL_SIZE
        return [
            (c, r)
            for c in range(int((lo + 180) / cs), int((hi + 180) / cs) + 1)
            for r in range(int((la +  90) / cs), int((ha +  90) / cs) + 1)
        ]
    def build(self, features):
        self.cells = {}
        for i, f in enumerate(features):
            for cell in self._cells(feature_bbox(f)):
                self.cells.setdefault(cell, []).append((i, f))

# Single-file startup cost: load original data, build 4 in-memory LODs, then build 4 indexes.
t0 = time.perf_counter()
startup_raw_features = load_raw_features()
startup_lods = build_all_lods_from_raw(startup_raw_features)
startup_indexes = {}

for name, features in startup_lods.items():
    idx = GridIndex()
    idx.build(features)
    startup_indexes[name] = idx

total_startup = time.perf_counter() - t0
print(f"Total startup time from one raw file (load + make LODs + index build): {total_startup:.2f}s")



Total startup time from one raw file (load + make LODs + index build): 8.17s


### Pain Point 2 — GeoJSON Is Verbose

GeoJSON is human-readable text. Every coordinate is stored as a decimal number string. A production mapping pipeline uses binary encoding (Mapbox Vector Tiles, MVT) which stores coordinates as integers relative to the tile origin — 5–10× smaller than equivalent GeoJSON and much faster to parse.

In [3]:
# Rough estimate: how large would our in-memory LODs be in a binary format?
# MVT stores coordinates as compact integers relative to the tile origin.
# GeoJSON stores them as decimal number strings plus punctuation.

GEOJSON_BYTES_PER_COORD = 16   # rough avg chars for [lon, lat] pair including punctuation
MVT_BYTES_PER_COORD     = 4    # rough 2 bytes each for x and y

for name, feats in lod_features_by_name.items():
    total_pts = sum(geometry_point_count(f["geometry"]) for f in feats)
    fc = {"type": "FeatureCollection", "features": feats}
    actual_mb  = len(json.dumps(fc).encode("utf-8")) / 1_000_000
    est_mvt_mb = total_pts * MVT_BYTES_PER_COORD / 1_000_000
    print(f"{name:<38} estimated GeoJSON: {actual_mb:.2f}MB   est MVT: {est_mvt_mb:.2f}MB")



coarse                                 estimated GeoJSON: 1.08MB   est MVT: 0.02MB
medium                                 estimated GeoJSON: 9.55MB   est MVT: 0.20MB
fine                                   estimated GeoJSON: 9.61MB   est MVT: 0.21MB
extra_fine                             estimated GeoJSON: 10.15MB   est MVT: 0.30MB


### Pain Point 3 — The Whole File Is Always Resident

To query the fine LOD for Paris, we load the entire `railroads_fine.geojson` into memory — including Australia, South America, and Russia. A tile system would read only the Paris tile from a database, never touching the rest.

Our grid index helps at query time, but the full file still had to load first.

### Pain Point 4 — No Partial Load or Streaming

When the user pans to a new region, we re-query the index immediately — but the data was all loaded at startup. A tile server streams only the tiles the user actually views. If the user never visits Australia, those tiles are never fetched.

## The Decision Inventory

Every system embeds design decisions. Here are ours, stated explicitly:

| Decision | What we chose | What we gave up |
|---|---|---|
| File format | GeoJSON (text) | Binary efficiency |
| Simplification algorithm | Douglas-Peucker via Shapely | Topology-preserving alternatives |
| LOD levels | 4 fixed levels | Continuous zoom-adaptive detail |
| Coarse filter | scalerank ≤ 4 | Coverage in scalerank 5+ regions |
| Spatial index | Uniform 10° grid | Adaptive indexes (R-tree, quadtree) |
| Culling granularity | Feature bbox | True geometry intersection |
| Transition policy | Fixed zoom thresholds | Hysteresis (implemented but not used in final viewer) |
| Memory model | All data loaded at startup | Lazy / tile-based loading |

None of these decisions are wrong. They are appropriate for a teaching system built from scratch. A production system makes different choices for different reasons.

## Exercise A

Measure the peak memory usage of the viewer-style startup after the single raw railroad file is loaded, four in-memory LOD feature sets are created, and four grid indexes are built.

How does this compare to creating the four in-memory LOD feature sets without building indexes?


In [4]:
# Measure peak memory: (a) building the four LODs in memory, (b) building LODs + indexes
# This version uses only data/ne_10m_railroads.geojson. It does not read data/lod files.

import gc
import tracemalloc


def load_all_lod_features():
    # Load the single raw GeoJSON file and create four LOD feature lists in memory.
    raw_features = load_raw_features()
    return build_all_lods_from_raw(raw_features)

# Case A: load the raw file and create the four LOD feature lists only
gc.collect()
tracemalloc.start()
features_only = load_all_lod_features()
current_load_only, peak_load_only = tracemalloc.get_traced_memory()
tracemalloc.stop()

total_features = sum(len(features) for features in features_only.values())
del features_only

gc.collect()

# Case B: load the raw file, create four LOD feature lists, and build one grid index per LOD
tracemalloc.start()
features_by_lod = load_all_lod_features()
indexes_by_lod = {}
for label, features in features_by_lod.items():
    idx = GridIndex()
    idx.build(features)
    indexes_by_lod[label] = idx
current_with_indexes, peak_with_indexes = tracemalloc.get_traced_memory()
tracemalloc.stop()

index_references = sum(
    len(items)
    for idx in indexes_by_lod.values()
    for items in idx.cells.values()
)

increase_mb = (peak_with_indexes - peak_load_only) / 1_000_000
ratio = peak_with_indexes / peak_load_only if peak_load_only else float("inf")

print(f"Peak memory, LOD features only:          {peak_load_only / 1_000_000:.1f} MB")
print(f"Peak memory, LOD features + indexes:    {peak_with_indexes / 1_000_000:.1f} MB")
print(f"Extra memory used by indexes:           {increase_mb:.1f} MB")
print(f"LOD + index memory / LOD-only memory:   {ratio:.2f}x")
print(f"Total in-memory LOD features:           {total_features:,}")
print(f"Total grid-index feature references:    {index_references:,}")

print("Interpretation:")
print(
    "The LOD-only case measures the cost of reading the single railroad file and creating the "
    "four simplified feature sets in memory. The index case uses more memory because each feature "
    "is additionally referenced inside one or more grid cells. The index improves viewport query "
    "speed, but it does not remove the startup cost of preparing data before the map becomes usable."
)



Peak memory, LOD features only:          310.3 MB
Peak memory, LOD features + indexes:    310.3 MB
Extra memory used by indexes:           0.0 MB
LOD + index memory / LOD-only memory:   1.00x
Total in-memory LOD features:           79,084
Total grid-index feature references:    82,687
Interpretation:
The LOD-only case measures the cost of reading the single railroad file and creating the four simplified feature sets in memory. The index case uses more memory because each feature is additionally referenced inside one or more grid cells. The index improves viewport query speed, but it does not remove the startup cost of preparing data before the map becomes usable.


## Exercise B

The Railroad LOD system is a handbuilt map-optimization pipeline for displaying railroad data without drawing the full original dataset at every zoom level. It solves the problem of making a large line dataset easier to load, query, and view interactively inside a notebook. The first major component is Douglas-Peucker simplification, which reduces the number of points in each railroad line while preserving its general shape. The second major component is the LOD pipeline, which writes four GeoJSON files at different detail levels: coarse, medium, fine, and extra fine. The third major component is viewport culling, which uses bounding boxes to test whether a feature overlaps the visible map area. The fourth major component is the uniform grid index, which places features into geographic grid cells so the viewer does not have to scan every feature during every pan or zoom. The LOD decision function connects these pieces by choosing the correct detail level based on zoom, and the live viewer uses that choice to display the selected features. The main performance tradeoff is that the system becomes faster during map interaction, but it still pays a startup cost because all LOD files are loaded and indexed first. Another tradeoff is that GeoJSON is easy to inspect and debug, but it is much larger and slower to parse than a binary tile format. If this system had to serve 10 million users, I would replace the all-in-memory GeoJSON workflow with prebuilt vector tiles, server-side tile storage, and lazy loading so each user receives only the tiles needed for the current viewport.

## Check Your Understanding

For a user on a slow connection, I would rank the pain points this way: **1) no partial load or streaming, 2) whole-file loading, 3) verbose GeoJSON format, and 4) startup cost**. No streaming is the biggest problem because the user has to wait for data they may never view, instead of receiving only the visible map area. Whole-file loading is closely related because it forces unnecessary data into memory, while verbose GeoJSON makes the download larger; startup cost matters too, but it is mostly the result of these earlier design choices rather than the root problem.